In [1]:
import sys
sys.path.append(r'N:\DE\CC Transaction Pipeline')
import cfg.config as config

from pyspark.sql import SparkSession
import os

os.environ['HADOOP_HOME'] = r'C:\Users\nisha\hadoop'

spark = SparkSession.builder \
    .appName("CCPipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.367") \
    .config("spark.hadoop.fs.s3a.access.key", config.ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", config.ACCESS_KEY_SECRET) \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()


In [2]:
# import pyspark
# print(pyspark.__version__)

In [3]:
# spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

In [4]:
brz_dim_customer = spark.read \
        .format('csv') \
        .option('header','True') \
        .load(os.path.join(config.LANDING_DIR, 'dim_customer.csv'))

brz_dim_customer.show()


+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+
|      first|     last|gender|              street|                city|state|  zip|    lat|              long|city_pop|                 job|       dob|           cust_uuid|
+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+
|   Jennifer|    Banks|     F|      561 Perry Cove|      Moravian Falls|   NC|28654|36.0788|          -81.1781|    3495|Psychologist, cou...|1988-03-09|8df71411-7ed9-484...|
|  Stephanie|     Gill|     F|43039 Riley Green...|              Orient|   WA|99160|48.8878|         -118.2105|     149|Special education...|1978-06-21|5d0d6615-62a5-443...|
|     Edward|  Sanchez|     M|594 White Dale Su...|          Malad City|   ID|83252|42.1808|          -112.262|    4154|Nature con

In [5]:
brz_dim_customer.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/brz/brz_dim_customer/')

In [6]:
brz_dim_merchant = spark.read \
        .format('csv') \
        .option('header','True') \
        .load(os.path.join(config.LANDING_DIR, 'dim_merchant.csv'))

brz_dim_merchant.show()


+--------------------+------------------+------------------+--------------------+
|            merchant|         merch_lat|        merch_long|          merch_uuid|
+--------------------+------------------+------------------+--------------------+
| fraud_Abbott-Rogahn|         26.460232|        -83.062399|f55a4f80-f9d2-41c...|
|fraud_Abbott-Steuber|         31.218339|        -81.606691|4dfca935-cd0b-42a...|
|fraud_Abernathy a...|          45.38473|       -121.637729|8aa32121-aa16-471...|
|   fraud_Abshire PLC|          37.62768|         -96.24989|ea16e3e8-8ef9-447...|
|fraud_Adams, Kova...|         49.807705|       -117.388555|6da32252-8d24-49c...|
| fraud_Adams-Barrows|         47.334898|        -85.664218|870dc211-14c1-48b...|
|fraud_Altenwerth,...|         34.625988|        -91.983465|15366c36-5c49-430...|
|fraud_Altenwerth-...|         45.716706|       -119.886246|352ffc57-3c10-487...|
| fraud_Ankunding LLC|29.889152000000003|        -97.003361|23929047-7dbe-474...|
|fraud_Ankunding

In [7]:
brz_dim_merchant.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/brz/brz_dim_merchant/')

In [8]:
brz_dim_card = spark.read \
        .format('csv') \
        .option('header','True') \
        .load(os.path.join(config.LANDING_DIR, 'dim_card.csv'))

brz_dim_card.show()


+-------------------+--------------------+-------------+---------------+----------+------------+
|             cc_num|       card_provider|card_category|expiration_date|issue_date|credit_limit|
+-------------------+--------------------+-------------+---------------+----------+------------+
|   2703186189652095|    American Express|    Signature|     2030-02-24|2025-02-25|       45000|
|       630423337322|       VISA 19 digit|         Gold|     2028-04-02|2024-04-03|       15000|
|     38859492057661|        JCB 16 digit|         Gold|     2026-08-26|2021-08-27|        6000|
|   3534093764340240|             Maestro|     Platinum|     2025-09-10|2022-09-11|       27500|
|    375534208663984|        JCB 15 digit|     Infinite|     2028-08-03|2024-08-04|      100000|
|   4767265376804500|       VISA 16 digit|     Infinite|     2026-06-03|2022-06-04|       90000|
|     30074693890476|       VISA 13 digit|         Gold|     2027-12-04|2022-12-05|        7000|
|   6011360759745864|Diners Cl

In [9]:
brz_dim_card.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/brz/brz_dim_card/')

In [10]:
brz_fact_transactions = spark.read \
        .format('csv') \
        .option('header','True') \
        .option('inferSchema','True') \
        .load(os.path.join(config.LANDING_DIR, 'fact_transactions.csv'))

brz_fact_transactions.show()


+---------------------+-------------------+--------------------+-------------+------+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+----------+---------+------------------+--------+--------------------+--------------------+
|trans_date_trans_time|             cc_num|            merchant|     category|   amt|      first|     last|gender|              street|                city|state|  zip|    lat|              long|city_pop|                 job|       dob|           trans_num| unix_time|merch_lat|        merch_long|is_fraud|          merch_uuid|           cust_uuid|
+---------------------+-------------------+--------------------+-------------+------+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+----------+---------+------------------+-----

In [11]:
brz_fact_transactions = brz_fact_transactions.withColumn('year',brz_fact_transactions['trans_date_trans_time'].substr(1,4)) \
                        .withColumn('month',brz_fact_transactions['trans_date_trans_time'].substr(6,2)) \
                        .withColumn('day',brz_fact_transactions['trans_date_trans_time'].substr(9,2))

brz_fact_transactions.show()

+---------------------+-------------------+--------------------+-------------+------+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+----------+---------+------------------+--------+--------------------+--------------------+----+-----+---+
|trans_date_trans_time|             cc_num|            merchant|     category|   amt|      first|     last|gender|              street|                city|state|  zip|    lat|              long|city_pop|                 job|       dob|           trans_num| unix_time|merch_lat|        merch_long|is_fraud|          merch_uuid|           cust_uuid|year|month|day|
+---------------------+-------------------+--------------------+-------------+------+-----------+---------+------+--------------------+--------------------+-----+-----+-------+------------------+--------+--------------------+----------+--------------------+----------+----

In [12]:
brz_fact_transactions.write \
    .format('parquet') \
    .mode('overwrite') \
    .partitionBy('year','month','day') \
    .save('s3a://cc-transaction-pipeline/brz/brz_fact_transactions/')